# 01 — Setup & data shards (Colab T4)

Clone **midi-llm** + **musicinstruct**, install dependencies, authenticate Hugging Face, and build **Stage S3 edit shards** for REMI, AMT, and Octuple (ABC disabled).

**Each notebook uses its own Colab runtime** (you cannot share a VM across notebook tabs). Run the **Sync to Drive** cell at the end so notebooks **02** and **03** can restore shards and the **same pilot manifest** in a new session.

**Runtime:** GPU → T4 (or better).

In [ ]:
# @title Configuration — edit if your forks differ
MIDI_LLM_REPO = "https://github.com/juliagsy/midi-llm.git"
MUSICINSTRUCT_REPO = "https://github.com/juliagsy/musicinstruct.git"
BRANCH = "main"  # or your feature branch

WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
RUN_ID = "pilot_v1"  # reuse in notebooks 02 and 03
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"
SHARDS_ROOT = f"{WORK}/data/shards/{RUN_ID}"

USE_DRIVE = True  # @param {type:"boolean"}  recommended for multi-notebook workflow
DRIVE_ROOT = "/content/drive/MyDrive/midi-llm"
DRIVE_SHARDS = f"{DRIVE_ROOT}/shards/{RUN_ID}"
DRIVE_PILOT = f"{DRIVE_ROOT}/pilot/{RUN_ID}"  # manifest + midi_in (notebook 03 needs same pilot as 01)

In [ ]:
# @title Hugging Face token (Llama 3.2 license required)
import os
from getpass import getpass

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF token (hf_...): ")

from huggingface_hub import login
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# @title Clone repos
!rm -rf {MIDI_LLM} {MUSICINSTRUCT}
!git clone --branch {BRANCH} --depth 1 {MIDI_LLM_REPO} {MIDI_LLM}
!git clone --branch {BRANCH} --depth 1 {MUSICINSTRUCT_REPO} {MUSICINSTRUCT}
%cd {MIDI_LLM}

In [ ]:
# @title Install midi-llm + musicinstruct (T4-friendly stack)
# Colab torchao breaks peft; LoRA does not need it
!pip uninstall -y torchao 2>/dev/null || true
!pip install -q -U "transformers>=4.44,<5" "accelerate>=0.33"
!pip install -q -e ".[repr,data,train]"
!pip install -q -e "{MUSICINSTRUCT}"
# Optional AMT arm (GitHub install)
!pip install -q "git+https://github.com/jthickstun/anticipation.git" mido

In [ ]:
# @title GPU check
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    raise RuntimeError("Enable GPU: Runtime → Change runtime type → T4 GPU")

In [ ]:
# @title Generate MIDI-Instruct pilot data (if not in repo)
import os
from pathlib import Path

manifest = Path(MUSICINSTRUCT) / "data/pilot/pilot.jsonl"
if not manifest.is_file():
    !cd {MUSICINSTRUCT} && musicinstruct generate-pilot --output-dir data/pilot --target 300
    !cd {MUSICINSTRUCT} && musicinstruct validate data/pilot/pilot.jsonl
    !cd {MUSICINSTRUCT} && musicinstruct self-test data/pilot/pilot.jsonl
else:
    print("Using existing manifest:", manifest)

In [ ]:
# @title Build S3 edit shards (train + validation) for all representations
import os
from pathlib import Path

os.makedirs(SHARDS_ROOT, exist_ok=True)
manifest = f"{MUSICINSTRUCT}/data/pilot/pilot.jsonl"
reprs = ["remi", "octuple", "amt"]

for split in ("train", "validation"):
    for repr_name in reprs:
        out = f"{SHARDS_ROOT}/edit_{repr_name}_{split}.jsonl"
        !midi-llm build-instruct-shard {manifest} --repr {repr_name} --output {out} --split {split}

print("Shards written to:", SHARDS_ROOT)
!ls -la {SHARDS_ROOT}

In [ ]:
# @title Validate shards (line counts + Octuple wire format)
import json
from pathlib import Path

reprs = ["remi", "octuple", "amt"]
manifest = Path(MUSICINSTRUCT) / "data/pilot/pilot.jsonl"
sample_midi = None
with manifest.open(encoding="utf-8") as handle:
    for line in handle:
        row = json.loads(line)
        if row.get("split") == "train":
            candidate = manifest.parent / row["midi_in"]
            if candidate.is_file():
                sample_midi = candidate
                break

for repr_name in reprs:
    shard = Path(SHARDS_ROOT) / f"edit_{repr_name}_train.jsonl"
    lines = shard.read_text(encoding="utf-8").strip().splitlines()
    record = json.loads(lines[0])
    completion = record["completion"]
    assert record["repr_name"] == repr_name, (repr_name, record.get("repr_name"))
    print(f"{repr_name}: {len(lines)} train examples, repr_name={record['repr_name']}")
    if repr_name == "octuple":
        assert "|" in completion and "[" not in completion, "bad octuple payload (expected a,b,c|d,e,f)"
    else:
        assert "[" not in completion, f"bad {repr_name} payload (expected flat integers)"

if sample_midi:
    !midi-llm roundtrip {sample_midi} --repr octuple --json
    !midi-llm token-stats {sample_midi} --repr octuple --no-bpe
else:
    print("Skipped roundtrip smoke (no train midi_in found)")

In [ ]:
# @title Sync shards + pilot to Google Drive (for notebooks 02/03 in a new runtime)
import shutil
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_SHARDS).parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SHARDS_ROOT, DRIVE_SHARDS, dirs_exist_ok=True)
    print("Synced shards →", DRIVE_SHARDS)
    print("Files:", sorted(p.name for p in Path(DRIVE_SHARDS).glob("*.jsonl")))
    pilot_src = Path(MUSICINSTRUCT) / "data/pilot"
    shutil.copytree(pilot_src, DRIVE_PILOT, dirs_exist_ok=True)
    print("Synced pilot →", DRIVE_PILOT)
else:
    print("Skipped Drive sync. Shards are only on this VM:", SHARDS_ROOT)
    print("Re-run notebook 01 in notebook 02/03, or enable USE_DRIVE here.")

In [ ]:
# @title Optional — GigaMIDI syntax shard (S0, no Lakh)
BUILD_GIGAMIDI = False  # @param {type:"boolean"}
GIGAMIDI_LIMIT = 200  # @param {type:"integer"}

if BUILD_GIGAMIDI:
    out = f"{SHARDS_ROOT}/syntax_remi_gigamidi.jsonl"
    !midi-llm build-gigamidi-shard --repr remi --output {out} --limit {GIGAMIDI_LIMIT}
else:
    print("Skipped GigaMIDI (set BUILD_GIGAMIDI=True to enable)")

In [ ]:
# @title Training time estimate on this GPU
import json
from midi_llm.config import load_config
from midi_llm.train.estimate import estimate_training

_seq_len = 1024  # match notebook 02 T4 override; protocol default is load_config("remi")["model"]["max_seq_len"]
for steps in (50, 300, 3000):
    report = estimate_training(repr_name="remi", steps=steps, seq_len=_seq_len, run_benchmark=False)
    est = report["estimate"]
    print(f"steps={steps}: ~{est['estimated_hours']} h ({est['seconds_per_step']} s/step, {est['notes']})")

## Next

1. Run **Sync to Drive** above (if `USE_DRIVE=True`).
2. **Disconnect** this runtime (optional).
3. Open **`02_train_lora_sft.ipynb`** in a **new** session (same `RUN_ID`, `USE_DRIVE=True`).